# 🐍 Day 07 · Python Basics — Files, Folders & Text Chunks (RAG Fundamentals)

> **Goal of this notebook:** RAG starts with *loading files*, *cutting them into chunks* and *finding the best chunks*. Do all three in plain Python before using LangChain.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Paths and folders with `pathlib`
2. Writing and reading text files
3. Finding files with `glob`
4. Cutting text into chunks
5. List comprehensions
6. Scoring chunks (a tiny retriever)
7. Building the final prompt

## 1. Paths and folders with `pathlib`
`Path` is the modern way to work with files and folders.

In [ ]:
from pathlib import Path

folder = Path("demo_docs")          # a path to a folder (it may not exist yet)
folder.mkdir(exist_ok=True)         # create it (no error if it already exists)

print(folder.exists())              # True
print(Path.cwd())                   # the folder this notebook is running in

## 2. Writing and reading text files

In [ ]:
(folder / "tesla.txt").write_text(
    "Tesla makes electric cars. Tesla was founded in 2003. Its CEO is Elon Musk."
, encoding="utf-8")
(folder / "nvidia.txt").write_text(
    "Nvidia designs GPUs. GPUs are used to train AI models. Nvidia was founded in 1993."
, encoding="utf-8")

text = (folder / "tesla.txt").read_text(encoding="utf-8")   # '/' joins a folder and a file name
print(text)

## 3. Finding files with `glob`
`1_ingestion_pipeline.py` loads every `*.txt` file in the `docs` folder. `glob("*.txt")` finds them.

In [ ]:
documents = []
for file in folder.glob("*.txt"):           # every file ending with .txt
    documents.append({"source": file.name, "text": file.read_text(encoding="utf-8")})

for doc in documents:
    print(doc["source"], "->", len(doc["text"]), "characters")

## 4. Cutting text into chunks
Long documents are cut into small **chunks** so we can find just the useful part.
`range(start, stop, step)` jumps forward `step` at a time.

In [ ]:
def chunk_words(text, size=6):
    words = text.split()
    chunks = []
    for start in range(0, len(words), size):     # 0, 6, 12, ...
        piece = words[start:start + size]        # take 'size' words
        chunks.append(" ".join(piece))
    return chunks

for c in chunk_words(documents[0]["text"]):
    print("•", c)

## 5. List comprehensions — a short way to build lists
```
[expression for item in list]                 # transform every item
[expression for item in list if condition]    # keep only some items
```

In [ ]:
numbers = [1, 2, 3, 4, 5]

squares = [n * n for n in numbers]
evens   = [n for n in numbers if n % 2 == 0]
print(squares, evens)

# Make ALL chunks from ALL documents, remembering where each came from
all_chunks = [
    {"source": doc["source"], "text": chunk}
    for doc in documents
    for chunk in chunk_words(doc["text"])
]
print(len(all_chunks), "chunks")

## 6. 🔗 A tiny retriever
A real retriever compares **embeddings**. Our tiny one simply counts how many question words appear in each chunk, then sorts the chunks by that score.

In [ ]:
def score(question, chunk_text):
    q_words = set(question.lower().split())       # set = unique words
    c_words = set(chunk_text.lower().replace(".", "").split())
    return len(q_words & c_words)                 # & = words in BOTH

question = "who is the ceo of tesla"

ranked = sorted(all_chunks, key=lambda c: score(question, c["text"]), reverse=True)
top_k = ranked[:2]                                # keep the best 2

for c in top_k:
    print(score(question, c["text"]), "|", c["source"], "|", c["text"])

## 7. Building the final prompt
The **G** in RAG: put the retrieved chunks into the prompt and let the LLM answer.

In [ ]:
context = "\n".join(c["text"] for c in top_k)

prompt = f'''Answer the question using ONLY the context below.

Context:
{context}

Question: {question}
'''
print(prompt)

In [ ]:
# Clean up the demo folder
import shutil
shutil.rmtree(folder)
print("demo_docs deleted ✅")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Use a list comprehension to make a list of the lengths of `["rag", "vector", "llm"]`.
2. Change `chunk_words` so each chunk has 4 words and print the chunks of the Nvidia text.
3. Use `sorted(..., key=len)` to sort `["banana", "kiwi", "apple"]` by length.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print([len(w) for w in ["rag", "vector", "llm"]])

nvidia = "Nvidia designs GPUs. GPUs are used to train AI models. Nvidia was founded in 1993."
for c in chunk_words(nvidia, size=4):
    print(c)

print(sorted(["banana", "kiwi", "apple"], key=len))
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Path | `Path('docs') / 'a.txt'` |
| Write / read file | `p.write_text(t), p.read_text()` |
| Find files | `folder.glob('*.txt')` |
| Step through a range | `range(0, len(words), 6)` |
| List comprehension | `[n * n for n in nums if n > 2]` |
| Common words | `set(a) & set(b)` |
| Sort by score | `sorted(items, key=f, reverse=True)` |
| Top k | `ranked[:2]` |

## ➡️ What's next?
You now know the Python used in **Day_07_RAG_Fundamentals**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀